In [49]:
import fitz
from tqdm.auto import tqdm
import os

pdf_path = "../dataset/constitutionalLaw.pdf"

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    
    cleaned_text = text.replace("\n", " ").replace('Error! No text of specified style in document.', ' ').replace("[...]", " ").strip()
    return cleaned_text

def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """
    Opens a PDF file, reads its text content page by page, and collects statistics.

    Parameters:
        pdf_path (str): The file path to the PDF document to be opened and read.

    Returns:
        list[dict]: A list of dictionaries, each containing the page number
        (adjusted), character count, word count, sentence count, token count, and the extracted text
        for each page.
    """
    doc = fitz.open(pdf_path)  # open a document
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):  # iterate the document pages
        text = page.get_text()  # get plain text encoded as UTF-8
        text = text_formatter(text)
        pages_and_texts.append({"page_number": page_number - 41,  # adjust page numbers since our PDF starts on page 42
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count_raw": len(text.split(". ")),
                                "page_token_count": len(text) / 4,  # 1 token = ~4 chars
                                "text": text})
    return pages_and_texts

if not os.path.exists(pdf_path):
    print("[INFO] File doesn't exist")
else:
    pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)

0it [00:00, ?it/s]

In [50]:
pages_and_texts[:2]

[{'page_number': -41,
  'page_char_count': 13,
  'page_word_count': 2,
  'page_sentence_count_raw': 1,
  'page_token_count': 3.25,
  'text': '[author name]'},
 {'page_number': -40,
  'page_char_count': 1062,
  'page_word_count': 207,
  'page_sentence_count_raw': 8,
  'page_token_count': 265.5,
  'text': "1     Elements     The thesis of this course is that there are two kinds of fidelity at the  core of the Supreme Court's elaboration of our Constitution — fidelity  to meaning and fidelity to role. Fidelity to meaning is the fidelity that  we all expect of any court or judge: it is the effort to preserve the  meaning, across context. Fidelity to role is the constraint on that  practice of preservation.  In this part, I introduce these separate elements. Section 1 begins with  a puzzle: Is Article V the exclusive mode by which the Constitution can  be amended? And if it is, then was Article XIII of the Articles of  Confederation? And if it was, then was the Constitution  constitutionall

In [15]:
pages_and_texts[990]

{'page_number': 949,
 'page_char_count': 4125,
 'page_word_count': 713,
 'page_sentence_count_raw': 33,
 'page_token_count': 1031.25,
 'text': '4: Fidelity on the Left 990  Although the policy arguments for extending marriage to same-sex couples may be compelling,  the legal arguments for requiring such an extension are not. The fundamental right to marry  does not include a right to make a State change its definition of marriage. And a State\'s  decision to maintain the meaning of marriage that has persisted in every culture throughout  human history can hardly be called irrational. In short, our Constitution does not enact any  one theory of marriage. The people of a State are free to expand marriage to include same-sex  couples, or to retain the historic definition.  Today, however, the Court takes the extraordinary step of ordering every State to license and  recognize same-sex marriage. Many people will rejoice at this decision, and I begrudge none  their celebration. But for thos

In [16]:
import random

random.sample(pages_and_texts, k=3)

[{'page_number': 127,
  'page_char_count': 3789,
  'page_word_count': 658,
  'page_sentence_count_raw': 59,
  'page_token_count': 947.25,
  'text': "2: Fidelity on the Right 168  states of destination and is not prohibited unless by other Constitutional provisions. [  … ]  Whatever their motive and purpose, regulations of commerce which do not infringe  some constitutional prohibition are within the plenary power conferred on Congress  by the Commerce Clause. Subject only to that limitation, presently to be considered,  we conclude that the prohibition of the shipment interstate of goods produced under  the forbidden substandard labor conditions is within the constitutional authority of  Congress.  In the more than a century which has elapsed since the decision of Gibbons v.  Ogden, these principles of constitutional interpretation have been so long and  repeatedly recognized by this Court as applicable to the Commerce Clause, that there  would be little occasion for repeating them now

In [17]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,13,2,1,3.25,[author name]
1,-40,1109,215,9,277.25,Error! No text of specified style in document....
2,-39,1835,370,6,458.75,1: Elements 2 Text and Context ...
3,-38,2002,405,20,500.50,Error! No text of specified style in document....
4,-37,3003,556,18,750.75,1: Elements 4 The first object of inquiry is...
...,...,...,...,...,...,...
1004,963,3943,679,38,985.75,4: Fidelity on the Left 1004 The majority's d...
1005,964,2923,513,21,730.75,Error! No text of specified style in document....
1006,965,3252,564,25,813.00,4: Fidelity on the Left 1006 dignity because ...
1007,966,3063,535,26,765.75,Error! No text of specified style in document....


In [18]:
# Get stats
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,1009.00,1009.00,1009.00,1009.00,1009.00
mean,463.00,3744.75,659.51,31.61,936.19
std,291.42,707.18,117.89,13.74,176.79
min,-41.00,0.00,1.00,1.00,0.00
25%,211.00,3445.00,610.00,22.00,861.25
50%,463.00,3866.00,675.00,29.00,966.50
75%,715.00,4187.00,731.00,38.00,1046.75
max,967.00,5027.00,910.00,121.00,1256.75


In [19]:
from spacy.lang.en import English

nlp = English()

nlp.add_pipe("sentencizer")

for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)
    
    # Make sure all sentences are strings
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]
    
    # Count the sentences 
    item["page_sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/1009 [00:00<?, ?it/s]

In [20]:
random.sample(pages_and_texts, k=1)

[{'page_number': -19,
  'page_char_count': 4133,
  'page_word_count': 729,
  'page_sentence_count_raw': 18,
  'page_token_count': 1033.25,
  'text': "1: Elements  22  saying,*406 'this constitution, and the laws of the United States, which shall be made  in pursuance thereof,' 'shall be the supreme law of the land,' [ … ] 'anything in the  constitution or laws of any state to the contrary notwithstanding.'  Among the enumerated powers, we do not find that of establishing a bank or creating  a corporation. But there is no phrase in the instrument which, like the articles of  confederation, excludes incidental or implied powers; and which requires that  everything granted shall be expressly and minutely described. Even the 10th  amendment, which was framed for the purpose of quieting the excessive jealousies  which had been excited, omits the word 'expressly,' and declares only, that the  powers 'not delegated to the United States, nor prohibited to the states, are reserved  to the state

In [21]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,1009.00,1009.00,1009.00,1009.00,1009.00,1009.00
mean,463.00,3744.75,659.51,31.61,936.19,27.52
std,291.42,707.18,117.89,13.74,176.79,8.53
min,-41.00,0.00,1.00,1.00,0.00,0.00
25%,211.00,3445.00,610.00,22.00,861.25,22.00
50%,463.00,3866.00,675.00,29.00,966.50,26.00
75%,715.00,4187.00,731.00,38.00,1046.75,33.00
max,967.00,5027.00,910.00,121.00,1256.75,64.00


# Chuncking

In [22]:
# Define split size to turn groups of sentences into chunks
num_sentence_chunk_size = 10 

# Create a function that recursively splits a list into desired sizes
def split_list(input_list: list, 
               slice_size: int) -> list[list[str]]:
    """
    Splits the input_list into sublists of size slice_size (or as close as possible).

    For example, a list of 17 sentences would be split into two lists of [[10], [7]]
    """
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

# Loop through pages and texts and split sentences into chunks
for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list=item["sentences"],
                                         slice_size=num_sentence_chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

  0%|          | 0/1009 [00:00<?, ?it/s]

In [23]:
# Sample an example from the group (note: many samples have only 1 chunk as they have <=10 sentences total)
random.sample(pages_and_texts, k=1)

[{'page_number': 287,
  'page_char_count': 2428,
  'page_word_count': 467,
  'page_sentence_count_raw': 33,
  'page_token_count': 607.0,
  'text': '2: Fidelity on the Right 328  of justice at the suit of individuals. This is fully discussed by writers on public law. It  is enough for us to declare its existence. The legislative department of a State  represents its polity and its will; and is called upon by the highest demands of natural  and political law to preserve justice and judgment, and to hold inviolate the public  obligations. Any departure from this rule, except for reasons most cogent, (of which  the legislature, and not the courts, is the judge,) never fails in the end to incur the  odium of the world, and to bring lasting injury upon the State itself. But to deprive  the legislature of the power of judging what the honor and safety of the State may  require, even at the expense of a temporary failure to discharge the public debts,  would be attended with greater evils than

In [24]:
# Create a DataFrame to get stats
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,1009.00,1009.00,1009.00,1009.00,1009.00,1009.00,1009.00
mean,463.00,3744.75,659.51,31.61,936.19,27.52,3.21
std,291.42,707.18,117.89,13.74,176.79,8.53,0.91
min,-41.00,0.00,1.00,1.00,0.00,0.00,0.00
25%,211.00,3445.00,610.00,22.00,861.25,22.00,3.00
50%,463.00,3866.00,675.00,29.00,966.50,26.00,3.00
75%,715.00,4187.00,731.00,38.00,1046.75,33.00,4.00
max,967.00,5027.00,910.00,121.00,1256.75,64.00,7.00


# Splitting Chunks

In [25]:
import re

# Split each chunk into its own item
pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]
        
        # Join the sentences together into a paragraph-like structure, aka a chunk (so they are a single string)
        joined_sentence_chunk = "".join(sentence_chunk).replace("  ", " ").strip()
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk) # ".A" -> ". A" for any full-stop/capital letter combo 
        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        # Get stats about the chunk
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4 # 1 token = ~4 characters
        
        pages_and_chunks.append(chunk_dict)

# How many chunks do we have?
len(pages_and_chunks)

  0%|          | 0/1009 [00:00<?, ?it/s]

3238

In [26]:
# View a random sample
random.sample(pages_and_chunks, k=1)

[{'page_number': 277,
  'sentence_chunk': '622, 37 L. Ed.463 (1893), and Cherokee Nation v. Southern Kansas R. Co., 135 U. S. 641, 657-659, 10 S. Ct.965, 34 L. Ed.295 (1890)). The fact that the Fifth Amendment requires the payment of just compensation when the Government exercises its power of eminent domain does not turn the taking into a commercial transaction between the landowner and the Government, let alone a government-compelled transaction between the landowner and a third party. 6 In an attempt to recast the individual mandate as a regulation of commercial activity, Justice GINSBURG suggests that "[a]n individual who opts not to purchase insurance from a private insurer can be seen as actively selecting another form of insurance: self-insurance."Post, at 2622. But "self-insurance" is, in this context, nothing more than a description of the failure to purchase insurance. Individuals are no more "activ[e] in the self-insurance market" when they fail to purchase insurance, ibid.,

In [27]:
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,3238.00,3238.00,3238.00,3238.00
mean,479.07,1150.50,189.79,287.62
std,289.33,570.62,94.93,142.66
min,-41.00,1.00,1.00,0.25
25%,226.00,775.00,127.00,193.75
50%,479.50,1128.50,186.00,282.12
75%,735.00,1497.00,245.00,374.25
max,967.00,4928.00,855.00,1232.00


In [28]:
# Show random chunks with under 30 tokens in length
min_token_length = 30
for row in df[df["chunk_token_count"] <= min_token_length].sample(5).iterrows():
    print(f'Chunk token count: {row[1]["chunk_token_count"]} | Text: {row[1]["sentence_chunk"]}')

Chunk token count: 4.0 | Text: Is it meant that
Chunk token count: 18.5 | Text: it is not our role to forbid it, or to pass upon its wisdom or fairness. D
Chunk token count: 22.25 | Text: The act also defines the term 'affecting commerce' section 2(7), 29 U. S. C. A. § 152(7):
Chunk token count: 22.5 | Text: at 1246.     Lee v. Washington Lee, Commissioner of Corrections of Alabama, et al. 4.2.5.2
Chunk token count: 24.75 | Text: That, on the other hand, 'a prolific source of dispute had been the maintenance by the railroads of


In [30]:
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
pages_and_chunks_over_min_token_len[:2]

[{'page_number': -40,
  'sentence_chunk': "Error!No text of specified style in document.1   Elements   The thesis of this course is that there are two kinds of fidelity at the core of the Supreme Court's elaboration of our Constitution — fidelity to meaning and fidelity to role. Fidelity to meaning is the fidelity that we all expect of any court or judge: it is the effort to preserve the meaning, across context. Fidelity to role is the constraint on that practice of preservation. In this part, I introduce these separate elements. Section 1 begins with a puzzle: Is Article V the exclusive mode by which the Constitution can be amended?And if it is, then was Article XIII of the Articles of Confederation?And if it was, then was the Constitution constitutionally ratified?And if it was, then how do we account for the background context that makes that ratification valid?",
  'chunk_char_count': 834,
  'chunk_word_count': 145,
  'chunk_token_count': 208.5},
 {'page_number': -40,
  'sentence_c

# Embedding chunks

In [32]:
from sentence_transformers import SentenceTransformer
# dimensions = 512
embedding_model = SentenceTransformer(model_name_or_path="mixedbread-ai/mxbai-embed-large-v1", 
                                      device="cpu")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/113k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [38]:
%%time

embedding_model.to("cuda")

for item in tqdm(pages_and_chunks_over_min_token_len):
    item["embedding"] = embedding_model.encode(item["sentence_chunk"])

  0%|          | 0/3152 [00:00<?, ?it/s]

CPU times: total: 1min 32s
Wall time: 2min 59s


In [51]:
# Turn text chunks into a single list
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]

In [52]:
%%time

# Embed all texts in batches
text_chunk_embeddings = embedding_model.encode(text_chunks,
                                               batch_size=32,
                                               convert_to_tensor=True)

text_chunk_embeddings

CPU times: total: 1min 45s
Wall time: 2min 53s


tensor([[-0.0439, -0.4659,  0.6479,  ...,  0.5028,  0.3789, -0.5409],
        [ 0.1588, -0.7958,  0.5648,  ...,  0.2567,  0.0368, -0.5021],
        [-0.1675, -0.3734,  0.4479,  ...,  0.0429, -0.0758,  0.1904],
        ...,
        [ 0.2830, -0.7314,  0.0943,  ..., -0.1528,  0.0866, -0.2628],
        [-0.1294, -0.2174, -0.0264,  ...,  0.2375,  0.0770, -0.0548],
        [ 0.2895, -0.4944,  0.0971,  ...,  0.6267,  0.0559, -0.2883]],
       device='cuda:0')

In [41]:
# Save embeddings to file
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
embeddings_df_save_path = "text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

In [42]:
# Import saved file and view
text_chunks_and_embedding_df_load = pd.read_csv(embeddings_df_save_path)
text_chunks_and_embedding_df_load.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-40,Error!No text of specified style in document.1...,834,145,208.50,[-0.04385296 -0.46593624 0.6479375 ... 0.50...
1,-40,Section 2 displays both fidelity to role and f...,248,44,62.00,[ 0.15875152 -0.79577005 0.5648467 ... 0.25...
2,-39,1: Elements 2 Text and Context Are amen...,1781,316,445.25,[-0.1674652 -0.37335205 0.4478985 ... 0.04...
3,-38,Error!No text of specified style in document.3...,1029,187,257.25,[-0.2554436 -0.5814156 1.4442168 ... 0.06...
4,-38,"No cause has been shown, and the present motio...",838,151,209.50,[-0.24289876 -0.22700739 0.48926654 ... 0.10...


In [43]:
import random

import torch
import numpy as np 
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

# Import texts and embedding df
text_chunks_and_embedding_df = pd.read_csv("text_chunks_and_embeddings_df.csv")

# Convert embedding column back to np.array (it got converted to string when it got saved to CSV)
text_chunks_and_embedding_df["embedding"] = text_chunks_and_embedding_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

# Convert texts and embedding df to list of dicts
pages_and_chunks = text_chunks_and_embedding_df.to_dict(orient="records")

# Convert embeddings to torch tensor and send to device (note: NumPy arrays are float64, torch tensors are float32 by default)
embeddings = torch.tensor(np.array(text_chunks_and_embedding_df["embedding"].tolist()), dtype=torch.float32).to(device)
embeddings.shape

C:\Users\chand\AppData\Local\Temp\ipykernel_13664\4062955739.py:13: DeprecationWarning: string or file could not be read to its end due to unmatched data; this will raise a ValueError in the future.
  text_chunks_and_embedding_df["embedding"] = text_chunks_and_embedding_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))


torch.Size([3152, 3])

In [44]:
text_chunks_and_embedding_df.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-40,Error!No text of specified style in document.1...,834,145,208.50,"[-0.04385296, -0.46593624, 0.6479375]"
1,-40,Section 2 displays both fidelity to role and f...,248,44,62.00,"[0.15875152, -0.79577005, 0.5648467]"
2,-39,1: Elements 2 Text and Context Are amen...,1781,316,445.25,"[-0.1674652, -0.37335205, 0.4478985]"
3,-38,Error!No text of specified style in document.3...,1029,187,257.25,"[-0.2554436, -0.5814156, 1.4442168]"
4,-38,"No cause has been shown, and the present motio...",838,151,209.50,"[-0.24289876, -0.22700739, 0.48926654]"


In [45]:
from sentence_transformers import util
# 1. Define the query
# Note: This could be anything. But since we're working with a nutrition textbook, we'll stick with nutrition-based queries.
query = "constitution law"
print(f"Query: {query}")

# 2. Embed the query to the same numerical space as the text examples 
# Note: It's important to embed your query with the same model you embedded your examples with.
query_embedding = embedding_model.encode(query, convert_to_tensor=True)

# 3. Get similarity scores with the dot product (we'll time this for fun)
from time import perf_counter as timer

start_time = timer()
dot_scores = util.dot_score(a=query_embedding, b=embeddings)[0]
end_time = timer()

print(f"Time take to get scores on {len(embeddings)} embeddings: {end_time-start_time:.5f} seconds.")

# 4. Get the top-k results (we'll keep this to 5)
top_results_dot_product = torch.topk(dot_scores, k=5)
top_results_dot_product

Query: constitution law


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x1024 and 3x3152)

In [34]:
# Define helper function to print wrapped text 
import textwrap

def print_wrapped(text, wrap_length=80):
    wrapped_text = textwrap.fill(text, wrap_length)
    print(wrapped_text)

In [35]:
print(f"Query: '{query}'\n")
print("Results:")
# Loop through zipped together scores and indicies from torch.topk
for score, idx in zip(top_results_dot_product[0], top_results_dot_product[1]):
    print(f"Score: {score:.4f}")
    # Print relevant sentence chunk (since the scores are in descending order, the most relevant chunk will be first)
    print("Text:")
    print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
    # Print the page number too so we can reference the textbook further (and check the results)
    print(f"Page number: {pages_and_chunks[idx]['page_number']}")
    print("\n")

Query: 'macronutrients functions'

Results:
Score: 0.6926
Text:
Macronutrients Nutrients that are needed in large amounts are called
macronutrients. There are three classes of macronutrients: carbohydrates,
lipids, and proteins. These can be metabolically processed into cellular energy.
The energy from macronutrients comes from their chemical bonds. This chemical
energy is converted into cellular energy that is then utilized to perform work,
allowing our bodies to conduct their basic functions. A unit of measurement of
food energy is the calorie. On nutrition food labels the amount given for
“calories” is actually equivalent to each calorie multiplied by one thousand. A
kilocalorie (one thousand calories, denoted with a small “c”) is synonymous with
the “Calorie” (with a capital “C”) on nutrition food labels. Water is also a
macronutrient in the sense that you require a large amount of it, but unlike the
other macronutrients, it does not yield calories. Carbohydrates Carbohydrates
are 

In [36]:
# Get GPU available memory
import torch
gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
gpu_memory_gb = round(gpu_memory_bytes / (2**30))
print(f"Available GPU memory: {gpu_memory_gb} GB")

Available GPU memory: 32 GB


In [37]:
torch.cuda.get_device_capability(0)[0]

7

In [38]:
# Note: the following is Gemma focused, however, there are more and more LLMs of the 2B and 7B size appearing for local use.
if gpu_memory_gb < 5.1:
    print(f"Your available GPU memory is {gpu_memory_gb}GB, you may not have enough memory to run a Gemma LLM locally without quantization.")
elif gpu_memory_gb < 8.1:
    print(f"GPU memory: {gpu_memory_gb} | Recommended model: Gemma 2B in 4-bit precision.")
    use_quantization_config = True 
    model_id = "google/gemma-2b-it"
elif gpu_memory_gb < 19.0:
    print(f"GPU memory: {gpu_memory_gb} | Recommended model: Gemma 2B in float16 or Gemma 7B in 4-bit precision.")
    use_quantization_config = False 
    model_id = "google/gemma-2b-it"
elif gpu_memory_gb > 19.0:
    print(f"GPU memory: {gpu_memory_gb} | Recommend model: Gemma 7B in 4-bit or float16 precision.")
    use_quantization_config = False 
    model_id = "google/gemma-7b-it"

print(f"use_quantization_config set to: {use_quantization_config}")
print(f"model_id set to: {model_id}")

GPU memory: 32 | Recommend model: Gemma 7B in 4-bit or float16 precision.
use_quantization_config set to: False
model_id set to: google/gemma-7b-it


In [47]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import is_flash_attn_2_available 

# 1. Create quantization config for smaller model loading (optional)
# Requires !pip install bitsandbytes accelerate, see: https://github.com/TimDettmers/bitsandbytes, https://huggingface.co/docs/accelerate/
# For models that require 4-bit quantization (use this if you have low GPU memory available)
from transformers import BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16)

# Bonus: Setup Flash Attention 2 for faster inference, default to "sdpa" or "scaled dot product attention" if it's not available
# Flash Attention 2 requires NVIDIA GPU compute capability of 8.0 or above, see: https://developer.nvidia.com/cuda-gpus
# Requires !pip install flash-attn, see: https://github.com/Dao-AILab/flash-attention 
if (is_flash_attn_2_available()) and (torch.cuda.get_device_capability(0)[0] >= 8):
  attn_implementation = "flash_attention_2"
else:
  attn_implementation = "sdpa"
print(f"[INFO] Using attention implementation: {attn_implementation}")

# 2. Pick a model we'd like to use (this will depend on how much GPU memory you have available)
#model_id = "google/gemma-7b-it"
model_id = model_id # (we already set this above)
print(f"[INFO] Using model_id: {model_id}")

# 3. Instantiate tokenizer (tokenizer turns text into numbers ready for the model) 
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_id)

# 4. Instantiate the model
llm_model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_id, 
                                                 torch_dtype=torch.float16, # datatype to use, we want float16
                                                 quantization_config=quantization_config if use_quantization_config else None,
                                                 low_cpu_mem_usage=False, # use full memory 
                                                 attn_implementation=attn_implementation) # which attention version to use

if not use_quantization_config: # quantization takes care of device setting automatically, so if it's not used, send model to GPU 
    llm_model.to("cuda")

[INFO] Using attention implementation: sdpa
[INFO] Using model_id: google/gemma-7b-it


tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.11G [00:00<?, ?B/s]

Gemma's activation function should be approximate GeLU and not exact GeLU.
Changing the activation function to `gelu_pytorch_tanh`.if you want to use the legacy `gelu`, edit the `model.config` to set `hidden_activation=gelu`   instead of `hidden_act`. See https://github.com/huggingface/transformers/pull/29402 for more details.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [48]:
llm_model

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 3072, padding_idx=0)
    (layers): ModuleList(
      (0-27): 28 x GemmaDecoderLayer(
        (self_attn): GemmaSdpaAttention(
          (q_proj): Linear(in_features=3072, out_features=4096, bias=False)
          (k_proj): Linear(in_features=3072, out_features=4096, bias=False)
          (v_proj): Linear(in_features=3072, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=3072, bias=False)
          (rotary_emb): GemmaRotaryEmbedding()
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=3072, out_features=24576, bias=False)
          (up_proj): Linear(in_features=3072, out_features=24576, bias=False)
          (down_proj): Linear(in_features=24576, out_features=3072, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): GemmaRMSNorm()
        (post_attention_layernorm): GemmaRMSNorm()
      )
    )
    (norm): Gemm

In [49]:
def get_model_num_params(model: torch.nn.Module):
    return sum([param.numel() for param in model.parameters()])

get_model_num_params(llm_model)

8537680896

In [50]:
def get_model_mem_size(model: torch.nn.Module):
    """
    Get how much memory a PyTorch model takes up.

    See: https://discuss.pytorch.org/t/gpu-memory-that-model-uses/56822
    """
    # Get model parameters and buffer sizes
    mem_params = sum([param.nelement() * param.element_size() for param in model.parameters()])
    mem_buffers = sum([buf.nelement() * buf.element_size() for buf in model.buffers()])

    # Calculate various model sizes
    model_mem_bytes = mem_params + mem_buffers # in bytes
    model_mem_mb = model_mem_bytes / (1024**2) # in megabytes
    model_mem_gb = model_mem_bytes / (1024**3) # in gigabytes

    return {"model_mem_bytes": model_mem_bytes,
            "model_mem_mb": round(model_mem_mb, 2),
            "model_mem_gb": round(model_mem_gb, 2)}

get_model_mem_size(llm_model)

{'model_mem_bytes': 17075361792,
 'model_mem_mb': 16284.33,
 'model_mem_gb': 15.9}

In [54]:
input_text = "Explain about macronutrients?"
print(f"Input text:\n{input_text}")

# Create prompt template for instruction-tuned model
dialogue_template = [
    {"role": "user",
     "content": input_text}
]

# Apply the chat template
prompt = tokenizer.apply_chat_template(conversation=dialogue_template,
                                       tokenize=False, # keep as raw text (not tokenized)
                                       add_generation_prompt=True)
print(f"\nPrompt (formatted):\n{prompt}")

Input text:
Explain about macronutrients?

Prompt (formatted):
<bos><start_of_turn>user
Explain about macronutrients?<end_of_turn>
<start_of_turn>model



In [55]:
%%time

# Tokenize the input text (turn it into numbers) and send it to GPU
input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")
print(f"Model input (tokenized):\n{input_ids}\n")

# Generate outputs passed on the tokenized input
# See generate docs: https://huggingface.co/docs/transformers/v4.38.2/en/main_classes/text_generation#transformers.GenerationConfig 
outputs = llm_model.generate(**input_ids,
                             max_new_tokens=256) # define the maximum number of new tokens to create
print(f"Model output (tokens):\n{outputs[0]}\n")

Model input (tokenized):
{'input_ids': tensor([[     2,      2,    106,   1645,    108,  74198,   1105, 186809, 184592,
         235336,    107,    108,    106,   2516,    108]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

Model output (tokens):
tensor([     2,      2,    106,   1645,    108,  74198,   1105, 186809, 184592,
        235336,    107,    108,    106,   2516,    108,  21404, 235269,   1517,
           603,    671,  15844,    576, 186809, 184592, 235292,    109,  12298,
          1695, 184592,    708,    573,   2149, 186809, 184592,    674,    573,
          2971,   7177,    577,   3658,   4134,    578,   2500,    578,  12158,
         29703, 235265,   2365,    708,  72780, 235269,  20361, 235269,    578,
         61926, 235265,   9573, 186809,   7208,    579,    919,    476,   2167,
          9408,   5449,    578,   6572,    476,   2167,   3619,    576,   4134,
           842,  14063, 235265,    109,    688,

In [56]:
# Decode the output tokens to text
outputs_decoded = tokenizer.decode(outputs[0])
print(f"Model output (decoded):\n{outputs_decoded}\n")

Model output (decoded):
<bos><bos><start_of_turn>user
Explain about macronutrients?<end_of_turn>
<start_of_turn>model
Sure, here is an explanation of macronutrients:

Macronutrients are the three macronutrients that the body uses to provide energy and build and repair tissues. They are carbohydrates, proteins, and fats. Each macronutrient has a different chemical structure and provides a different amount of energy per gram.

**Carbohydrates:**

* Carbohydrates are the body's main source of energy. They are broken down into glucose, which is then used for energy.
* Carbohydrates are found in foods such as bread, pasta, rice, potatoes, fruits, and vegetables.
* Carbohydrates provide 4 kcal per gram.

**Proteins:**

* Proteins are used to build and repair tissues, produce enzymes and hormones, and transport nutrients throughout the body.
* Proteins are found in foods such as meat, fish, eggs, dairy products, beans, lentils, and nuts.
* Proteins provide 4 kcal per gram.

**Fats:**

* Fats 